# Colab: baseline → SFT → GRPO (in short bursts) → sweep

Runs the training pipeline with **early feedback at every step** so you can bail out or tweak the reward if things look bad, without waiting for the full 4h GRPO to finish.

**Runtime:** GPU (T4 works, V100/A100 much faster).

**Iteration pattern for GRPO:**
1. Train 50 steps.
2. Eval on val split (~2 min).
3. If reward is climbing, run the next 50 steps starting from the last checkpoint.
4. If it's flat or going down, stop, edit `RewardConfig` coefficients (or change GRPO hyperparams), and restart.

## 1. Clone repo + install deps

Repo is private. Colab will prompt for username (`Andrii238`) and password → paste a **personal access token** (from https://github.com/settings/tokens, `repo` scope), NOT your GitHub password.

In [ ]:
import os
if os.path.isdir('ml-project'):
    !cd ml-project && git pull
else:
    !git clone https://github.com/Andrii238/ml-project.git
%cd ml-project

In [ ]:
!pip install -q -r requirements-colab.txt
# Colab's preinstalled bitsandbytes has a mixed-version state
# (backends/cpu/ops.py imports has_avx512bf16 from functional.py where
# it doesn't exist). Force a clean reinstall of bnb + accelerate.
!pip install -q --upgrade --force-reinstall bitsandbytes accelerate
print('\n>>> RESTART RUNTIME NOW (Runtime > Restart runtime), then re-run from the *next* cell. <<<')

In [ ]:
# GPU + lib sanity.
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')
import transformers, trl, peft
print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)

In [ ]:
# Repo unit tests — 91/91 should pass.
!python -m mini_factorio.tests

## 2. STOP-CHECK: baseline eval (policy_0 = raw Qwen)

Confirms Qwen loads + generates + parsing/apply/reward pipeline works. Expected ~2-5 min (first run also downloads the model).

**If parse_ok_rate is 0 or valid_rate looks off, stop here.** The prompt or edit schema is broken; fix before training anything.

In [ ]:
from training.evaluate import evaluate_checkpoints, rows_to_table, save_results

baseline = evaluate_checkpoints(
    [{'name': 'policy_0', 'adapter': None}],
    samples_per_layout=2,
    max_new_tokens=1024,
    temperature=0.8,
)
print(rows_to_table(baseline))

## 3. SFT (Stage 1) — quick check after

~5-15 min on T4. Saves adapter to `ckpts/sft`.

In [ ]:
from training.train_sft import train as sft_train, SFTConfig

sft_train(SFTConfig(
    output_dir='ckpts/sft',
    epochs=3,
    per_device_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    load_in_4bit=False,
))

**STOP-CHECK:** post-SFT eval. `policy_1_sft` should score noticeably better than `policy_0`. If not, SFT didn't take — check the SFT dataset size, LR, or epochs.

In [ ]:
post_sft = evaluate_checkpoints(
    [{'name': 'policy_0',     'adapter': None},
     {'name': 'policy_1_sft', 'adapter': 'ckpts/sft'}],
    samples_per_layout=2,
)
print(rows_to_table(post_sft))

## 4. GRPO — 50 steps at a time with intermediate evals

Each block trains 50 GRPO steps, saves an adapter, evaluates it on the val split. Total ~30-60 min per block on T4.

**After each block, decide:**
- If `mean_reward` and `mean_green_science` are climbing, run the next block starting from the last GRPO checkpoint.
- If flat or dropping, **STOP** and either:
  - Edit `mini_factorio/reward.py::RewardConfig` defaults (change a coefficient), then restart from `ckpts/sft`.
  - Change GRPO hyperparams below (LR, β, temperature, G).

In [ ]:
from training.train_grpo import train as grpo_train, GRPOConfig

def run_grpo_burst(start_adapter, out_dir, steps=50):
    grpo_train(GRPOConfig(
        sft_adapter=start_adapter,
        output_dir=out_dir,
        num_generations=8,
        max_new_tokens=1024,
        beta=0.04,
        learning_rate=5e-5,
        max_steps=steps,
        per_device_batch_size=2,
        gradient_accumulation_steps=4,
        load_in_4bit=False,
    ))
    print(f'GRPO burst done → {out_dir}')

def quick_eval(adapter_paths):
    specs = [{'name': 'policy_0',     'adapter': None},
             {'name': 'policy_1_sft', 'adapter': 'ckpts/sft'}]
    for p in adapter_paths:
        specs.append({'name': p.split('/')[-1], 'adapter': p})
    r = evaluate_checkpoints(specs, samples_per_layout=2)
    print(rows_to_table(r))
    return r

In [ ]:
# ---- GRPO burst 1: steps 1..50, starting from SFT.
run_grpo_burst(start_adapter='ckpts/sft', out_dir='ckpts/grpo_050', steps=50)
quick_eval(['ckpts/grpo_050']);

In [ ]:
# ---- GRPO burst 2: steps 51..100, starting from grpo_050.
# ONLY RUN IF burst 1 showed improvement.
run_grpo_burst(start_adapter='ckpts/grpo_050', out_dir='ckpts/grpo_100', steps=50)
quick_eval(['ckpts/grpo_050', 'ckpts/grpo_100']);

In [ ]:
# ---- GRPO burst 3.
run_grpo_burst(start_adapter='ckpts/grpo_100', out_dir='ckpts/grpo_150', steps=50)
quick_eval(['ckpts/grpo_050', 'ckpts/grpo_100', 'ckpts/grpo_150']);

In [ ]:
# ---- GRPO burst 4.
run_grpo_burst(start_adapter='ckpts/grpo_150', out_dir='ckpts/grpo_200', steps=50)
quick_eval(['ckpts/grpo_050', 'ckpts/grpo_100', 'ckpts/grpo_150', 'ckpts/grpo_200']);

## 5. Full final eval (samples_per_layout=4)

Only run this once you're satisfied with GRPO training. Uses more samples per layout for tighter stds.

In [ ]:
import os

grpo_ckpts = sorted(d for d in os.listdir('ckpts') if d.startswith('grpo_'))
specs = [
    {'name': 'policy_0',     'adapter': None},
    {'name': 'policy_1_sft', 'adapter': 'ckpts/sft'},
] + [{'name': d, 'adapter': f'ckpts/{d}'} for d in grpo_ckpts]

results = evaluate_checkpoints(specs, samples_per_layout=4,
                                max_new_tokens=1024, temperature=0.8)
print(rows_to_table(results))
save_results(results, 'ckpts/eval_results.json')

## 6. Plots

In [ ]:
import matplotlib.pyplot as plt

rows = [r.as_row() for r in results]
xs = list(range(len(rows)))
means = [r['mean_reward'] for r in rows]
stds  = [r['std_reward']  for r in rows]
gs    = [r['mean_green_science'] for r in rows]
names = [r['name'] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].errorbar(xs, means, yerr=stds, fmt='o-', capsize=4)
axes[0].set_xticks(xs); axes[0].set_xticklabels(names, rotation=30, ha='right')
axes[0].set_ylabel('mean composite reward'); axes[0].set_title('Reward vs checkpoint')
axes[0].grid(True, alpha=0.3)

axes[1].plot(xs, gs, 's-', color='C1')
axes[1].set_xticks(xs); axes[1].set_xticklabels(names, rotation=30, ha='right')
axes[1].set_ylabel('mean green science /s'); axes[1].set_title('Green science vs checkpoint')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ckpts/plots.png', dpi=150)
plt.show()

# Per-iteration deltas.
deltas = [means[i+1] - means[i] for i in range(len(means)-1)]
print('\nPer-iteration reward deltas:')
for i, d in enumerate(deltas):
    print(f'  {names[i]:20s} → {names[i+1]:20s}: Δ = {d:+.3f}')
if len(deltas) > 1:
    print(f'\nΔ_final = {deltas[-1]:+.3f}   max Δ_intermediate = {max(deltas[:-1]):+.3f}')